In [2]:
import pandas as pd

df = pd.read_csv("zomato.csv")

df.head()
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 17 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   url                          51717 non-null  object
 1   address                      51717 non-null  object
 2   name                         51717 non-null  object
 3   online_order                 51717 non-null  object
 4   book_table                   51717 non-null  object
 5   rate                         43942 non-null  object
 6   votes                        51717 non-null  int64 
 7   phone                        50509 non-null  object
 8   location                     51696 non-null  object
 9   rest_type                    51490 non-null  object
 10  dish_liked                   23639 non-null  object
 11  cuisines                     51672 non-null  object
 12  approx_cost(for two people)  51371 non-null  object
 13  reviews_list                 51

In [4]:
df = df.drop(['url', 'address', 'phone'], axis=1)

In [5]:
df = df[df['rate'].notnull()]

df = df[df['rate'] != 'NEW']
df = df[df['rate'] != '-']

df['rate'] = df['rate'].apply(lambda x: str(x).split('/')[0])
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')

In [6]:
df['rate'].describe()

count    41665.000000
mean         3.700449
std          0.440513
min          1.800000
25%          3.400000
50%          3.700000
75%          4.000000
max          4.900000
Name: rate, dtype: float64

In [7]:
df['rate'].isnull().sum()

np.int64(0)

In [8]:
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str)
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].str.replace(',', '')
df['approx_cost(for two people)'] = pd.to_numeric(df['approx_cost(for two people)'], errors='coerce')

In [9]:
df['dish_liked'] = df['dish_liked'].fillna('Not Available')
df['cuisines'] = df['cuisines'].fillna('Unknown')
df['rest_type'] = df['rest_type'].fillna('Unknown')
df['location'] = df['location'].fillna('Unknown')

In [10]:
df.isnull().sum()
df.shape

(41665, 14)

In [11]:
df['approx_cost(for two people)'].isnull().sum()

np.int64(247)

In [13]:
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].fillna(
    df['approx_cost(for two people)'].median()
)

In [14]:
df['approx_cost(for two people)'].isnull().sum()

np.int64(0)

In [15]:
df['cost_per_person'] = df['approx_cost(for two people)'] / 2

In [16]:
df['price_category'] = pd.cut(df['cost_per_person'],
                             bins=[0,200,500,1000,5000],
                             labels=['Low','Medium','High','Luxury'])

In [17]:
df['rating_category'] = pd.cut(df['rate'],
                              bins=[0,2.5,3.5,4.5,5],
                              labels=['Poor','Average','Good','Excellent'])

In [18]:
df['cuisine_count'] = df['cuisines'].apply(lambda x: len(str(x).split(',')))

In [19]:
df['online_order'] = df['online_order'].map({'Yes':1, 'No':0})

In [20]:
df.head()

,name,online_order,book_table,rate,votes,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city),cost_per_person,price_category,rating_category,cuisine_count
0,Jalsa,1,Yes,4.1,775,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800.0,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari,400.0,Medium,Good,3
1,Spice Elephant,1,No,4.1,787,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800.0,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari,400.0,Medium,Good,3
2,San Churro Cafe,1,No,3.8,918,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800.0,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari,400.0,Medium,Good,3
3,Addhuri Udupi Bhojana,0,No,3.7,88,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300.0,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari,150.0,Low,Good,2
4,Grand Village,0,No,3.8,166,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600.0,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari,300.0,Medium,Good,2


In [21]:
###Do expensive restaurants really have better ratings?
df.groupby('price_category')['rate'].mean().sort_values(ascending=False)

C:\Users\diash\AppData\Local\Temp\ipykernel_12404\2125327.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('price_category')['rate'].mean().sort_values(ascending=False)


price_category
High      4.130434
Luxury    4.123501
Medium    3.708270
Low       3.578870
Name: rate, dtype: float64

In [22]:
location_stats = df.groupby('location').agg({
    'name':'count',
    'rate':'mean'
}).sort_values(by='name', ascending=False)

location_stats.head(10)

,name,rate
location,,
BTM,3930,3.573740
Koramangala 5th Block,2319,4.005821
HSR,2019,3.672164
Indiranagar,1847,3.828154
JP Nagar,1717,3.675306
Jayanagar,1643,3.780280
Whitefield,1582,3.621618
Marathahalli,1443,3.541927
Bannerghatta Road,1235,3.507449


In [23]:
df.groupby('online_order')['rate'].mean()

online_order
0    3.65907
1    3.72244
Name: rate, dtype: float64

In [24]:
df['cuisines'].value_counts().head(10)

cuisines
North Indian                           2158
North Indian, Chinese                  1980
South Indian                           1234
Cafe                                    642
Bakery, Desserts                        615
Biryani                                 609
South Indian, North Indian, Chinese     561
Desserts                                551
Fast Food                               517
Chinese                                 410
Name: count, dtype: int64

In [25]:
df.groupby('cuisines')['rate'].mean().sort_values(ascending=False).head(10)

cuisines
Continental, North Indian, Italian, South Indian, Finger Food            4.900000
Healthy Food, Salad, Mediterranean                                       4.900000
Asian, Chinese, Thai, Momos                                              4.900000
North Indian, European, Mediterranean, BBQ                               4.800000
Asian, Mediterranean, North Indian, BBQ                                  4.800000
Continental, North Indian, Chinese, European, BBQ, Finger Food, Asian    4.800000
European, Mediterranean, North Indian, BBQ                               4.789474
American, Tex-Mex, Burger, BBQ, Mexican                                  4.750000
Italian, American, Pizza                                                 4.700000
Chinese, American, Continental, Italian, North Indian                    4.700000
Name: rate, dtype: float64

In [26]:
location_stats = df.groupby('location').agg({
    'name':'count',
    'rate':'mean'
}).reset_index()

location_stats['expansion_score'] = (
    location_stats['name'] * 0.6 +
    location_stats['rate'] * 100 * 0.4
)

location_stats.sort_values(by='expansion_score', ascending=False).head(10)

,location,name,rate,expansion_score
0,BTM,3930,3.573740,2500.949618
44,Koramangala 5th Block,2319,4.005821,1551.632859
22,HSR,2019,3.672164,1358.286578
27,Indiranagar,1847,3.828154,1261.326151
29,JP Nagar,1717,3.675306,1177.212231
31,Jayanagar,1643,3.780280,1137.011199
88,Whitefield,1582,3.621618,1094.064728
55,Marathahalli,1443,3.541927,1007.477062
3,Bannerghatta Road,1235,3.507449,881.297976
45,Koramangala 6th Block,1077,3.778087,797.323491
